<a href="https://colab.research.google.com/github/LexerPatty/RAG/blob/main/Adaptive_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Adaptive RAG means a Retrieval-Augmented Generation (RAG) system that automatically adjusts how it searches, retrieves, and answers based on the user's question.**


**Imagine you're asking a librarian two different types of questions:**
1️⃣ Simple Question

You: “Who is the CEO of Google?”

Normal librarian:
Searches the database, finds one document, answers immediately.

Adaptive librarian (Adaptive RAG):
Recognizes the question is simple → retrieves ONLY 1–2 documents → answers fast.

**2️⃣ Complex Question**

You: “Compare Google’s CEO leadership style with Microsoft’s CEO and suggest which company will grow faster.”

Now the librarian needs:

Multiple sources

Historical comparisons

Market analysis

Detailed reasoning

**Adaptive RAG does this:**

✓ Retrieves more documents (maybe 20)
✓ Uses different retrieval types (semantic search + web search + vector search)
✓ Breaks question into smaller parts
✓ Uses deeper reasoning to generate accurate output

A normal RAG system wouldn’t adjust — it would retrieve the same number of documents and often give an incomplete answer.

In [ ]:
!pip install langchain langchain-groq chromadb tiktoken sentence-transformers unstructured langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, but you have opentelemetry-exporter-otlp-proto-common 1.38.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-proto==1.37.0, but you have opentelemetry-proto 1.38.0 which is incompatib

In [ ]:
import os

from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [ ]:
from langchain_community.document_loaders import UnstructuredURLLoader

# Example website with Goa travel info
url = "https://www.holidify.com/places/goa/sightseeing-and-things-to-do.html"

loader = UnstructuredURLLoader(urls=[url])
docs = loader.load()

print("Website Loaded Successfully!")
print("Sample Content:\n")
print(docs[0].page_content[:500])   # show preview


Website Loaded Successfully!
Sample Content:

Goa

Big Vagator Beach, one of the many beaches of Goa

Calangute Beach

Dudhsagar Falls

Fort Aguada



Get Holiday Package Offers

You can get a customised offer for your trip duration, dates, and group size

View Packages

View Packages

Cruise in Goa

Basilica of Bom Jesus

Anjuna Beach

Palolem Beach

Water Sports in Goa

Baga Beach

Places To Visit In Goa

"Beaches, Bazaars, and Portuguese Heritage"



30 out of 745 Places in India

Best Time: October to March Read More

4.9 /5

Read full 


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant")

In [ ]:
from langchain_community.vectorstores import Chroma
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings

# Simple embedding wrapper
class MiniLMEmbedder(Embeddings):
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

# Load embedder
embedder = MiniLMEmbedder()

# Convert docs → plain text
texts = [d.page_content for d in docs]

# Create vector DB
vectordb = Chroma.from_texts(texts=texts, embedding=embedder)

print("Vector DB Created!")


Vector DB Created!


In [ ]:
def normal_rag(query):
    retriever = vectordb.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(query)

    context = "\n".join([d.page_content for d in docs])

    context = context[:3000]

    prompt = f"""
    Use ONLY the information in the context.

    Context:
    {context}

    Question: {query}

    Answer:
    """

    res= llm.invoke(prompt)
    return res.content

In [ ]:
def adaptive_retriever(query):
    # Short question = simple → fewer docs
    if len(query.split()) < 7:
        k = 2
    else:
        k = 5   # complex query → more docs

    retriever = vectordb.as_retriever(search_kwargs={"k": k})
    return retriever.invoke(query)



In [ ]:
def adaptive_rag(query):
    docs = adaptive_retriever(query)

    context = "\n".join([d.page_content for d in docs])

    context = context[:3000]

    prompt = f"""
    Use ONLY the information in the context.

    Context:
    {context}

    Question: {query}

    Answer:
    """

    res= llm.invoke(prompt)
    return res.content


In [ ]:
query = "What is the website about?"

print("===== NORMAL RAG RESULT =====")
print(normal_rag(query))

print("\n===== ADAPTIVE RAG RESULT =====")
print(adaptive_rag(query))

===== NORMAL RAG RESULT =====
The website is about Goa, a state in India, and provides information about its various tourist attractions, beaches, waterfalls, forts, and heritage sites, as well as travel packages and hotel deals.

===== ADAPTIVE RAG RESULT =====
The website appears to be about Goa, an Indian state known for its beaches, heritage, and scenic beauty. It provides information about various places to visit in Goa, including beaches, forts, waterfalls, and churches. The website also offers holiday packages and tour options for visitors, allowing them to plan and book their trip to Goa.
